## Importing Libraries

In [1]:
from ollama import chat
import glob
from tqdm import tqdm
import os
import json
import re
from groq import Groq

## Setting up files

In [2]:
GENERATION_MODEL = "qwen3:1.7b"

GENERATION_MODEL = "qwen3:8b" 
GROQ_MODEL = "openai/gpt-oss-120b"

GROQ_KEY = os.getenv("GROQ_API_KEY")
CLIENT = Groq(api_key=GROQ_KEY)

TYPE_LLM = True # True - local, False - groq

FILES_CONV = glob.glob("../Test_Files/Clinical_trials/Criteria_extracted/clinical-trial-extracted*.txt")
GOLD_FILES = glob.glob("../Test_Files/Clinical_trials/GT-clinical-trial_*.json")

PROMPT_CONV_FILE = "./prompts/criteria_conversion/criteria-conversion_prompt.txt"
SYS_PROMPT_CONV_FILE = "./prompts/criteria_conversion/sys_criteria-conversion_prompt.txt"

OUTPUT_CONV_DIR = "./llm-outputs/criteria-conversion/"
OUTPUT_CONV_FILE = "experiment"

print(f"Found the following files for conversion - {FILES_CONV}")
print(f"Found the following golden diaries {GOLD_FILES}")

Found the following files for conversion - ['../Test_Files/Clinical_trials/Criteria_extracted\\clinical-trial-extracted_e1.txt', '../Test_Files/Clinical_trials/Criteria_extracted\\clinical-trial-extracted_e10.txt', '../Test_Files/Clinical_trials/Criteria_extracted\\clinical-trial-extracted_e11.txt', '../Test_Files/Clinical_trials/Criteria_extracted\\clinical-trial-extracted_e2.txt', '../Test_Files/Clinical_trials/Criteria_extracted\\clinical-trial-extracted_e3.txt', '../Test_Files/Clinical_trials/Criteria_extracted\\clinical-trial-extracted_e4.txt', '../Test_Files/Clinical_trials/Criteria_extracted\\clinical-trial-extracted_e5.txt', '../Test_Files/Clinical_trials/Criteria_extracted\\clinical-trial-extracted_e6.txt', '../Test_Files/Clinical_trials/Criteria_extracted\\clinical-trial-extracted_e7.txt', '../Test_Files/Clinical_trials/Criteria_extracted\\clinical-trial-extracted_e8.txt', '../Test_Files/Clinical_trials/Criteria_extracted\\clinical-trial-extracted_e9.txt']
Found the following

## Setting up environment

In [3]:
## Setting evironment
def set_env(prompt_file, sys_prompt_file,output_dir):
    with open(prompt_file,"r", encoding="utf-8") as p:
        base_prompt = p.read()
        
    with open(sys_prompt_file, "r", encoding = "utf-8") as sp:
        sys_prompt = sp.read()

    os.makedirs(output_dir,exist_ok=True)

    count = 0

    for path in os.listdir(output_dir):
        if os.path.isfile(os.path.join(output_dir, path)):
            count += 1
    
    return base_prompt, sys_prompt, count

base_prompt_conv, sys_prompt_conv, count_conv_exp = set_env(PROMPT_CONV_FILE, SYS_PROMPT_CONV_FILE, OUTPUT_CONV_DIR)

## Criteria Conversion
In this second phase the already extracted criteria in natural language of a given clinical trial will be converted into logical rules that way allowing the deterministic matching of patients with the clinical trial

In [4]:
pbar = tqdm(total=len(FILES_CONV), desc="Processing trials for criteria conversion")

for file in FILES_CONV:
    with open(file,"r", encoding="utf-8") as f:
        text_arr = [t.strip() for t in f.readlines() if t.strip()]
        text = " ".join(text_arr)
        
        print(f"processing file: {file}")
        
        prompt = base_prompt_conv.replace("{{CRITERIA_TEXT}}",text)
        
        if TYPE_LLM:
            stream = chat(
                model=GENERATION_MODEL,
                messages=[
                    {
                        "role": "system",
                        "content": sys_prompt_conv
                    },
                    {
                        "role": "user", 
                        "content": prompt
                        }
                    ],
                stream=True,
                options={"num_ctx": 32000}
                )
            
            llm_output = ""
            for chunk in stream:
                llm_output += chunk["message"]["content"]
                
        elif not TYPE_LLM:
            stream = CLIENT.chat.completions.create(
                model= GROQ_MODEL,
                messages=[
                    {
                        "role": "system",
                        "content": sys_prompt_conv
                    },
                    {
                        "role": "user",
                        "content": prompt
                    }
                ],
                temperature=0
            )
            
            llm_output = stream.choices[0].message.content

        with open(f"{OUTPUT_CONV_DIR}{OUTPUT_CONV_FILE}-{count_conv_exp}.txt","a",encoding="utf-8") as o:
            o.write(f"Ouput for file {file}\n")
            o.write(f"{llm_output}\n\n")
            print(f"Saved LLM output on {OUTPUT_CONV_FILE}-{count_conv_exp}")
            
        
        print("\n")                                                                                                                                                                                                                                                             
        
        pbar.update(1)
        
pbar.close()

Processing trials for criteria conversion:   0%|          | 0/11 [00:00<?, ?it/s]

processing file: ../Test_Files/Clinical_trials/Criteria_extracted\clinical-trial-extracted_e1.txt


Processing trials for criteria conversion:   9%|▉         | 1/11 [39:11<6:31:52, 2351.26s/it]

Saved LLM output on experiment-4


processing file: ../Test_Files/Clinical_trials/Criteria_extracted\clinical-trial-extracted_e10.txt


Processing trials for criteria conversion:  18%|█▊        | 2/11 [1:22:57<6:16:58, 2513.15s/it]

...

Saved LLM output on experiment-4


processing file: ../Test_Files/Clinical_trials/Criteria_extracted\clinical-trial-extracted_e8.txt


Processing trials for criteria conversion:  91%|█████████ | 10/11 [5:52:37<30:28, 1828.69s/it] 

Saved LLM output on experiment-4


processing file: ../Test_Files/Clinical_trials/Criteria_extracted\clinical-trial-extracted_e9.txt


Processing trials for criteria conversion: 100%|██████████| 11/11 [6:22:35<00:00, 2086.89s/it]

Saved LLM output on experiment-4




## Evaluation
In this phase the pipeline of extraction will be evaluated in 2 different fields:
- Correct classification (inclusion/exclusion)
- Logic correctness of rules

In [5]:
for gold_file in GOLD_FILES:
    with open(gold_file,"r",encoding="utf-8") as gf, \
         open(f"{OUTPUT_EXTR_DIR}{OUTPUT_EXTR_FILE}-{1}.txt","r",encoding="utf-8") as out_extr, \
         open(f"{OUTPUT_CONV_DIR}{OUTPUT_CONV_FILE}-{2}.txt","r",encoding="utf-8") as out_conv:
        
        curr_gf_trial = gold_file.split('_')[3].split('.')[0]
        
        print("Current trial: ", curr_gf_trial)

        data_gf = json.load(gf)

        out_extr_arr = [t.strip() for t in out_extr.readlines() if t.strip()]
        out_conv_arr = [t.strip() for t in out_conv.readlines() if t.strip()]

        out_extr_text = " ".join(out_extr_arr)
        out_conv_text = " ".join(out_conv_arr)

        outputs_extraction = out_extr_text.split("Ouput for file ")
        outputs_extraction.pop(0)

        outputs_conversion = out_conv_text.split("Ouput for file ")
        outputs_conversion.pop(0)
        
        print("outputs_extraction: ", outputs_extraction)
        print("outputs_conversion: ", outputs_conversion)
        
        for output_c, output_e in zip(outputs_conversion, outputs_extraction):
            if curr_gf_trial not in output_c or curr_gf_trial not in output_e:
                continue

            # Extract JSON safely
            json_conv_match = re.search(r"\{.*\}", output_c, flags=re.DOTALL)
            if not json_conv_match:
                print("No JSON found for", curr_gf_trial)
                continue
            
            json_extr_match = re.search(r"\{.*\}", output_e, flags=re.DOTALL)
            if not json_extr_match:
                print("No JSON found for", curr_gf_trial)
                continue
            
            output_conv_json = json.loads(json_conv_match.group(0))
            output_extr_json = json.loads(json_extr_match.group(0))

            print("Golden truth data ", data_gf)
            print("Output of the LLM (conversion) ", output_conv_json)
            print("Output of the LLM (extraction) ", output_extr_json)
            
            # Correct classification (inclusion/exclusion)
            
            gf_inclusion = set([c.strip().lower() for c in data_gf["inclusion_criteria"]])
            gf_exclusion = set([c.strip().lower() for c in data_gf["exclusion_criteria"]])

            llm_inclusion = set([c.strip().lower() for c in output_extr_json["inclusion_criteria"]])
            llm_exclusion = set([c.strip().lower() for c in output_extr_json["exclusion_criteria"]])
            
            correct = 0
            total = 0

            for crit in llm_inclusion:
                total += 1
                if any(crit in g or g in crit for g in gf_inclusion):
                    correct += 1

            for crit in llm_exclusion:
                total += 1
                if any(crit in g or g in crit for g in gf_exclusion):
                    correct += 1

            accuracy = correct / total if total > 0 else 0

            print("Correct classification:", correct, "/", total)
            print("Accuracy:", round(accuracy, 4))

NameError: name 'OUTPUT_EXTR_DIR' is not defined